# FastAPI Middleware & Error Handling

Covers custom middleware with `BaseHTTPMiddleware`, custom exception classes, exception handlers, and background tasks. All examples use `TestClient`.

In [ ]:
# !pip install fastapi httpx

from fastapi import FastAPI, Request, Response
from fastapi.testclient import TestClient
from starlette.middleware.base import BaseHTTPMiddleware
import time

# --- Custom middleware: request timing + request ID ---
class TimingMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request: Request, call_next):
        start = time.perf_counter()
        response: Response = await call_next(request)
        duration_ms = (time.perf_counter() - start) * 1000
        response.headers["X-Process-Time"] = f"{duration_ms:.2f}ms"
        return response

class RequestIDMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request: Request, call_next):
        import uuid
        request_id = str(uuid.uuid4())[:8]
        response = await call_next(request)
        response.headers["X-Request-ID"] = request_id
        return response

app = FastAPI()
app.add_middleware(TimingMiddleware)
app.add_middleware(RequestIDMiddleware)

@app.get("/ping")
def ping():
    return {"message": "pong"}

client = TestClient(app)
r = client.get("/ping")
print("Body:", r.json())
print("X-Process-Time:", r.headers.get("x-process-time"))
print("X-Request-ID:", r.headers.get("x-request-id"))

In [ ]:
# --- Custom exception classes and handlers ---
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient

# Custom exception hierarchy
class AppException(Exception):
    def __init__(self, status_code: int, detail: str, error_code: str = None):
        self.status_code = status_code
        self.detail = detail
        self.error_code = error_code or "APP_ERROR"

class NotFoundException(AppException):
    def __init__(self, resource: str, resource_id):
        super().__init__(404, f"{resource} with id={resource_id} not found", "NOT_FOUND")

class PermissionException(AppException):
    def __init__(self, action: str):
        super().__init__(403, f"Not allowed to {action}", "FORBIDDEN")

app2 = FastAPI()

# Register exception handler
@app2.exception_handler(AppException)
async def app_exception_handler(request: Request, exc: AppException):
    return JSONResponse(
        status_code=exc.status_code,
        content={"error": exc.error_code, "detail": exc.detail, "path": str(request.url.path)}
    )

ITEMS = {1: "Widget", 2: "Gadget"}

@app2.get("/items/{item_id}")
def get_item(item_id: int, admin: bool = False):
    if not admin and item_id == 2:
        raise PermissionException("view item 2")
    item = ITEMS.get(item_id)
    if not item:
        raise NotFoundException("Item", item_id)
    return {"id": item_id, "name": item}

client2 = TestClient(app2)

r = client2.get("/items/1")
print("Found:", r.json())

r = client2.get("/items/99")
print("Not found:", r.status_code, r.json())

r = client2.get("/items/2")
print("Forbidden:", r.status_code, r.json())

r = client2.get("/items/2?admin=true")
print("Admin access:", r.json())

In [ ]:
# --- Background tasks ---
from fastapi import FastAPI, BackgroundTasks
from fastapi.testclient import TestClient
import time

app3 = FastAPI()
task_log = []  # Shared log to observe background task execution

def send_welcome_email(email: str, username: str):
    """Simulated slow email send — runs after response is returned."""
    time.sleep(0.05)  # Simulate I/O
    task_log.append(f"Email sent to {email} for user {username}")
    print(f"  [BG] Email sent to {email}")

def log_signup(username: str):
    task_log.append(f"Signup logged for {username}")
    print(f"  [BG] Signup logged for {username}")

@app3.post("/register")
def register(username: str, email: str, background_tasks: BackgroundTasks):
    # These run after the response is sent
    background_tasks.add_task(send_welcome_email, email, username)
    background_tasks.add_task(log_signup, username)
    return {"message": f"User {username} registered", "status": "background tasks queued"}

client3 = TestClient(app3)

print("Registering user...")
r = client3.post("/register?username=alice&email=alice@example.com")
print("Response:", r.json())
print("Task log:", task_log)

In [ ]:
# --- Validation error handler + 422 customization ---
from fastapi import FastAPI
from fastapi.exceptions import RequestValidationError
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient
from pydantic import BaseModel, field_validator

app4 = FastAPI()

@app4.exception_handler(RequestValidationError)
async def validation_exception_handler(request, exc: RequestValidationError):
    errors = []
    for error in exc.errors():
        errors.append({
            "field": " -> ".join(str(loc) for loc in error["loc"]),
            "message": error["msg"],
            "type": error["type"]
        })
    return JSONResponse(status_code=422, content={"validation_errors": errors})

class Product(BaseModel):
    name: str
    price: float
    quantity: int

    @field_validator("price")
    @classmethod
    def price_must_be_positive(cls, v):
        if v <= 0:
            raise ValueError("Price must be positive")
        return v

@app4.post("/products")
def create_product(product: Product):
    return {"created": product.model_dump()}

client4 = TestClient(app4)

# Valid
r = client4.post("/products", json={"name": "Widget", "price": 9.99, "quantity": 5})
print("Valid:", r.json())

# Missing field + bad type
r = client4.post("/products", json={"name": "Widget", "price": "not-a-number"})
print("Invalid:", r.status_code)
for err in r.json()["validation_errors"]:
    print(" ", err)

# Negative price (custom validator)
r = client4.post("/products", json={"name": "Widget", "price": -5.0, "quantity": 1})
print("Negative price:", r.status_code, r.json())